# Full 3D Baseline Sweep — GBM vs All Available 3D Methods

This notebook runs a side-by-side comparison of **GBM-3D** against **all currently available 3D baseline implementations** on the same synthetic 3D volume.

Methods included (when successfully importable):
- GBM-XZ (D-1 pillars)
- Greedy 3D
- Uniform Grid 3D
- PSO 3D
- Monte Carlo 3D
- Random 3D
- Simulated Annealing 3D
- Genetic 3D

In [ ]:
import numpy as np
import pandas as pd
import traceback

print("Generating synthetic 3D volume...")

In [ ]:
# Small synthetic 3D barn volume (LW=2 for speed)
lw = 2
W, H_img, D = 100 * lw, 100, 12
x = np.linspace(-20, 20, W)
y = np.linspace(0.4, 12, D)
z = np.linspace(-70, 70, H_img)

X, Y, Z = np.meshgrid(x, y, z, indexing="ij")
C = 5e-6 + 3.5e-6 * np.exp(-((X)**2 + (Z)**2 + (Y-5)**2) / 70)

df = pd.DataFrame({
    "X": X.ravel(),
    "Y": Y.ravel(),
    "Z": Z.ravel(),
    "Carbon": C.ravel()
})

inside = np.ones(W * H_img * D, dtype=bool)
global_mean = float(C.mean())

print(f"3D volume: {W}x{H_img}x{D}, global mean = {global_mean:.3e}")

In [ ]:
# Dictionary to store results
results = {}

def safe_run(name, func, **kwargs):
    try:
        res = func(**kwargs)
        if isinstance(res, (list, tuple)) and len(res) >= 1:
            loss = float(res[0])
        else:
            loss = float(res)
        results[name] = loss
        print(f"{name:22s}: {loss:.3e}")
    except Exception as e:
        results[name] = f"ERROR: {type(e).__name__}"
        print(f"{name:22s}: ERROR - {e}")

In [ ]:
print("\nRunning 3D baseline sweep (k=5)...\n")

# GBM 3D
try:
    from gbm.core import find_optimal_k_points_gbm_3D
    safe_run("GBM-XZ (D-1)", find_optimal_k_points_gbm_3D,
             nodes_df=df, barn_inside_points=inside, k=5, in_CO2_avg=global_mean,
             cross_section="XZ", barn_LW_ratio=lw, epochs=5, sampling_budget=600)
except Exception as e:
    results["GBM-XZ (D-1)"] = f"IMPORT ERROR: {e}"
    print("GBM 3D import failed")

In [ ]:
# Greedy 3D
try:
    from gbm.baselines.greedy import find_optimal_k_points_greedy_3D
    safe_run("Greedy 3D", find_optimal_k_points_greedy_3D,
             nodes_df=df, barn_inside_points=inside, k=5, in_CO2_avg=global_mean,
             barn_LW_ratio=lw, sampling_budget=600)
except Exception as e:
    results["Greedy 3D"] = f"IMPORT ERROR: {e}"
    print("Greedy 3D import failed")

In [ ]:
# Uniform 3D
try:
    from gbm.baselines.uniform import find_optimal_k_points_uniform_grid_search_3D
    safe_run("Uniform Grid 3D", find_optimal_k_points_uniform_grid_search_3D,
             nodes_df=df, barn_inside_points=inside, k=5, in_CO2_avg=global_mean,
             barn_LW_ratio=lw, max_subset_evals=800)
except Exception as e:
    results["Uniform Grid 3D"] = f"IMPORT ERROR: {e}"
    print("Uniform 3D import failed")

In [ ]:
# PSO 3D
try:
    from gbm.baselines.PSO import find_optimal_k_points_pso_3D
    safe_run("PSO 3D", find_optimal_k_points_pso_3D,
             nodes_df=df, barn_inside_points=inside, k=5, in_CO2_avg=global_mean,
             barn_LW_ratio=lw, epochs=30)
except Exception as e:
    results["PSO 3D"] = f"IMPORT ERROR: {e}"
    print("PSO 3D import failed")

In [ ]:
# Monte Carlo 3D
try:
    from gbm.baselines.monte_carlo import find_optimal_k_points_monte_carlo_3D
    res = find_optimal_k_points_monte_carlo_3D(df, inside, 5, global_mean, barn_LW_ratio=lw, max_epochs=20)
    results["Monte Carlo 3D"] = float(res[0])
    print(f"Monte Carlo 3D      : {results['Monte Carlo 3D']:.3e}")
except Exception as e:
    results["Monte Carlo 3D"] = f"ERROR: {e}"
    print(f"Monte Carlo 3D failed: {e}")

In [ ]:
# Random 3D
try:
    from gbm.baselines.random import find_optimal_k_points_random_search_3D
    safe_run("Random 3D", find_optimal_k_points_random_search_3D,
             nodes_df=df, barn_inside_points=inside, k=5, in_CO2_avg=global_mean,
             barn_LW_ratio=lw, epochs=200)
except Exception as e:
    results["Random 3D"] = f"IMPORT ERROR: {e}"
    print("Random 3D import failed")

In [ ]:
# Simulated Annealing 3D
try:
    from gbm.baselines.simulated_annealing import find_optimal_k_points_simulated_annealing_3D
    safe_run("SA 3D", find_optimal_k_points_simulated_annealing_3D,
             nodes_df=df, barn_inside_points=inside, k=5, in_CO2_avg=global_mean,
             barn_LW_ratio=lw, epochs=200)
except Exception as e:
    results["SA 3D"] = f"IMPORT ERROR: {e}"
    print("SA 3D import failed")

In [ ]:
# Genetic 3D
try:
    from gbm.baselines.genetic import find_optimal_k_points_advanced_genetic_algorithm_3D
    safe_run("Genetic 3D", find_optimal_k_points_advanced_genetic_algorithm_3D,
             nodes_df=df, barn_inside_points=inside, k=5, in_CO2_avg=global_mean,
             barn_LW_ratio=lw, episodes=15)
except Exception as e:
    results["Genetic 3D"] = f"IMPORT ERROR: {e}"
    print("Genetic 3D import failed")

In [ ]:
print("\n=== Summary (lower loss is better) ===")
for name, val in sorted(results.items(), key=lambda x: (isinstance(x[1], (int,float)), x[1] if isinstance(x[1], (int,float)) else 999)):
    print(f"{name:22s}: {val}")